# 05 — Executive Dashboard: Fleet & Route Performance Analytics
Renders an in-notebook HTML dashboard using `displayHTML()` with KPI cards and summary tables.

In [ ]:
from pyspark.sql import functions as F


In [ ]:
# ── Pull KPI data ────────────────────────────────────────────────────────────
kpi = spark.sql("SELECT * FROM vw_executive_summary").collect()[0]

total_deliveries   = f"{kpi['total_deliveries']:,}"
on_time_rate       = f"{kpi['avg_on_time_rate_pct']:.1f}%"
avg_delay          = f"{kpi['avg_delay_hrs']:.2f} hrs"
total_km           = f"{int(kpi['total_km_driven']):,} km"
avg_load           = f"{kpi['avg_load_utilisation_pct']:.1f}%"

# Depot count
depot_count = spark.table("gold_depot_scorecards").count()
total_fuel  = spark.table("gold_depot_scorecards") \
    .agg(F.round(F.sum("total_fuel_cost_gbp"), 0).alias("t")).collect()[0]["t"]
fuel_cost   = f"£{int(total_fuel):,}"

print("KPIs collected.")


In [ ]:
# ── Top late routes ──────────────────────────────────────────────────────────
late_rows = (
    spark.table("gold_route_analysis")
    .orderBy(F.col("avg_delay_hrs").desc())
    .select("origin", "destination", "route_type", "total_deliveries",
            "on_time_rate_pct", "avg_delay_hrs")
    .limit(8)
    .collect()
)

late_html = ""
for r in late_rows:
    rate_color = "#d32f2f" if r["on_time_rate_pct"] < 80 else "#f57c00" if r["on_time_rate_pct"] < 90 else "#388e3c"
    late_html += f"""<tr>
        <td>{r['origin']} → {r['destination']}</td>
        <td>{r['route_type']}</td>
        <td>{r['total_deliveries']:,}</td>
        <td style='color:{rate_color};font-weight:bold'>{r['on_time_rate_pct']:.1f}%</td>
        <td>{r['avg_delay_hrs']:.2f}</td>
    </tr>"""


In [ ]:
# ── Fuel efficiency by vehicle type ─────────────────────────────────────────
fuel_rows = (
    spark.table("gold_fleet_summary")
    .groupBy("vehicle_type")
    .agg(
        F.count("vehicle_id").alias("vehicles"),
        F.round(F.avg("avg_load_utilisation_pct"), 1).alias("avg_load_pct"),
        F.round(F.avg("on_time_rate_pct"), 1).alias("on_time_pct"),
        F.round(F.sum("total_km_driven"), 0).alias("total_km"),
    )
    .orderBy(F.col("total_km").desc())
    .collect()
)

fuel_html = ""
for r in fuel_rows:
    fuel_html += f"""<tr>
        <td>{r['vehicle_type']}</td>
        <td>{r['vehicles']}</td>
        <td>{r['avg_load_pct']:.1f}%</td>
        <td>{r['on_time_pct']:.1f}%</td>
        <td>{int(r['total_km']):,}</td>
    </tr>"""


In [ ]:
# ── Render Dashboard ─────────────────────────────────────────────────────────
html = f"""
<style>
  body{{font-family:'Segoe UI',Arial,sans-serif;background:#f5f5f5;padding:24px;}}
  h1{{color:#1565c0;font-size:1.6em;margin-bottom:4px;}}
  .subtitle{{color:#666;margin-bottom:20px;font-size:.9em;}}
  .kpi-grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:16px;margin-bottom:28px;}}
  .kpi{{background:#fff;border-radius:10px;padding:18px 20px;box-shadow:0 2px 8px rgba(0,0,0,.07);}}
  .kpi .label{{font-size:.78em;color:#888;text-transform:uppercase;letter-spacing:.05em;}}
  .kpi .value{{font-size:1.9em;font-weight:700;color:#1565c0;margin-top:4px;}}
  .section{{background:#fff;border-radius:10px;padding:20px;box-shadow:0 2px 8px rgba(0,0,0,.07);margin-bottom:24px;}}
  .section h3{{margin:0 0 14px;color:#333;font-size:1em;}}
  table{{width:100%;border-collapse:collapse;font-size:.88em;}}
  th{{background:#1565c0;color:#fff;padding:8px 12px;text-align:left;font-weight:600;}}
  td{{padding:7px 12px;border-bottom:1px solid #eee;}}
  tr:hover td{{background:#f0f4ff;}}
</style>
<h1>🚛 Fleet & Route Performance Analytics</h1>
<div class='subtitle'>Transportation & Logistics | 90-Day Operating Period</div>

<div class='kpi-grid'>
  <div class='kpi'><div class='label'>Total Deliveries</div><div class='value'>{total_deliveries}</div></div>
  <div class='kpi'><div class='label'>On-Time Rate</div><div class='value' style='color:#388e3c'>{on_time_rate}</div></div>
  <div class='kpi'><div class='label'>Avg Delay</div><div class='value' style='color:#f57c00'>{avg_delay}</div></div>
  <div class='kpi'><div class='label'>Total Distance</div><div class='value'>{total_km}</div></div>
  <div class='kpi'><div class='label'>Avg Load Utilisation</div><div class='value'>{avg_load}</div></div>
  <div class='kpi'><div class='label'>Total Fuel Cost</div><div class='value' style='color:#d32f2f'>{fuel_cost}</div></div>
</div>

<div class='section'>
  <h3>🔴 Top Routes by Average Delay (worst first)</h3>
  <table>
    <tr><th>Route</th><th>Type</th><th>Deliveries</th><th>On-Time %</th><th>Avg Delay (hrs)</th></tr>
    {late_html}
  </table>
</div>

<div class='section'>
  <h3>🚚 Fleet Performance by Vehicle Type</h3>
  <table>
    <tr><th>Type</th><th>Vehicles</th><th>Avg Load %</th><th>On-Time %</th><th>Total KM</th></tr>
    {fuel_html}
  </table>
</div>
"""

displayHTML(html)